<a href="https://colab.research.google.com/github/RafaelMuniz94/RaruRevival/blob/main/Museu_do_Raru_Cutter_overlay_50pct_hex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Museu do Raru — Cutter com overlay e numeração de casos

Notebook para a **Fase 2** do Museu do Raru.

Ele usa a estrutura já existente do projeto:

```text
RaruRevival/
├── Lives/
├── Transcriptions/
├── Models/
├── Logs/
├── Temp/
├── Checkpoints/
├── Manifests/
└── Cuts/
```

## O que este notebook faz

1. Monta o Google Drive.
2. Cria automaticamente `Manifests/` e `Cuts/` caso não existam.
3. Cria também a pasta de controle de casos em `Checkpoints/MuseuCaseRegistry/`.
4. Procura primeiro os vídeos em `RaruRevival/Lives/`.
5. Se não houver nenhum vídeo, encerra sem procurar manifests.
6. Para cada vídeo, procura um JSON com o mesmo nome-base em `RaruRevival/Manifests/`.
7. Valida o manifesto e os timestamps.
8. Gera um MP4 por momento em `Cuts/<nome da live>/`.
9. Escreve overlays no vídeo:
   - **canto inferior esquerdo**: `live`, `game` e, quando `origin == VIEWER`, a frase `Momento escolhido por {indicated_by}`;
   - cada um desses campos fica em **uma linha física própria**;
   - o bloco é redimensionado automaticamente para nunca ultrapassar **50% da largura nem 50% da altura do vídeo**;
   - **canto superior esquerdo**: `Museu do RARU Caso: #{numero_caso}`, também limitado a 50% da largura/altura.
10. Mantém um registro persistente de casos em arquivo texto, no formato:

```text
numero_caso >> live >> game >> id >> origin >> data_do_registro
```

## Requisitos do manifest

Na raiz do JSON:

- `project`
- `live`
- `game`
- `moments`

Em cada momento:

- `id`
- `rank`
- `title`
- `start`
- `end`
- `origin`

Se `origin` for `VIEWER`, o campo `indicated_by` também deve existir.


## Cores dos overlays

As cores do texto são parametrizadas na primeira célula usando HEX:

```python
BOTTOM_OVERLAY_FONT_COLOR_HEX = "#FFFFFF"
CASE_OVERLAY_FONT_COLOR_HEX = "#FFFFFF"
```

É possível usar `#RRGGBB` ou `RRGGBB`.


In [ ]:
# ============================================================
# CÉLULA 1 — DRIVE, ESTRUTURA E CONFIGURAÇÃO
# ============================================================

from google.colab import drive
from pathlib import Path
from datetime import datetime
import os
import shutil


drive.mount("/content/drive")

# ------------------------------------------------------------
# RAIZ DO PROJETO
# ------------------------------------------------------------

RARU_REVIVAL_ROOT = Path("/content/drive/MyDrive/RaruRevival")

# Pastas já existentes
LIVES_DIR = RARU_REVIVAL_ROOT / "Lives"
TRANSCRIPTIONS_DIR = RARU_REVIVAL_ROOT / "Transcriptions"
MODELS_DIR = RARU_REVIVAL_ROOT / "Models"
LOGS_DIR = RARU_REVIVAL_ROOT / "Logs"
TEMP_DIR = RARU_REVIVAL_ROOT / "Temp"
CHECKPOINTS_DIR = RARU_REVIVAL_ROOT / "Checkpoints"

# Pastas do Museu
MANIFESTS_DIR = RARU_REVIVAL_ROOT / "Manifests"
CUTS_DIR = RARU_REVIVAL_ROOT / "Cuts"
CASE_REGISTRY_DIR = CHECKPOINTS_DIR / "MuseuCaseRegistry"
CASE_REGISTRY_FILE = CASE_REGISTRY_DIR / "case_registry.txt"

# Cria a estrutura, inclusive as novas pastas
PROJECT_DIRS = [
    LIVES_DIR,
    TRANSCRIPTIONS_DIR,
    MODELS_DIR,
    LOGS_DIR,
    TEMP_DIR,
    CHECKPOINTS_DIR,
    MANIFESTS_DIR,
    CUTS_DIR,
    CASE_REGISTRY_DIR,
]

for directory in PROJECT_DIRS:
    directory.mkdir(parents=True, exist_ok=True)

if not CASE_REGISTRY_FILE.exists():
    CASE_REGISTRY_FILE.touch()

# ------------------------------------------------------------
# CONFIGURAÇÕES
# ------------------------------------------------------------

# Qualidade: menor CRF = maior qualidade e arquivo maior.
# 14 é uma configuração de altíssima qualidade.
VIDEO_CRF = 14

# ultrafast, veryfast, fast, medium, slow, slower, veryslow
PRESET = "slow"

AUDIO_BITRATE = "320k"

# True = substitui cortes existentes.
OVERWRITE = True

# True = cria um ZIP de cada live processada.
CREATE_ZIP = True

# Se True, um erro em um momento não impede os demais.
CONTINUE_ON_ERROR = True

VIDEO_EXTENSIONS = {
    ".mp4", ".mkv", ".mov", ".m4v", ".webm", ".avi"
}

# Fonte usada pelo FFmpeg para os overlays.
# Se este arquivo não existir no ambiente, o FFmpeg tentará usar a fonte padrão.
OVERLAY_FONT_PATH = Path("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf")

# Aparência do overlay
#
# Cada bloco de texto pode ocupar NO MÁXIMO 25% da largura
# e 25% da altura do vídeo. O tamanho da fonte é calculado
# automaticamente para respeitar esses limites.
OVERLAY_MAX_WIDTH_RATIO = 0.50
OVERLAY_MAX_HEIGHT_RATIO = 0.50

# Tamanho desejado para 1080p. O código reduz automaticamente
# quando o texto não couber no limite de 25%.
OVERLAY_BASE_FONT_SIZE_1080 = 36

# Pode reduzir até 1 px em casos extremos para nunca ultrapassar
# o limite definido. Na prática, títulos normais ficam bem maiores.
OVERLAY_MIN_FONT_SIZE = 1

OVERLAY_MARGIN_RATIO = 0.0125
OVERLAY_LINE_SPACING_RATIO = 0.25
OVERLAY_BOX_OPACITY = 0.45

# ------------------------------------------------------------
# CORES DOS TEXTOS (HEX)
# ------------------------------------------------------------
#
# Aceita formatos:
#   "#FFFFFF"
#   "FFFFFF"
#
# O código valida e converte automaticamente para o formato
# aceito pelo FFmpeg.
#
# Texto inferior esquerdo:
# Live / Game / Momento escolhido por...
BOTTOM_OVERLAY_FONT_COLOR_HEX = "#FFFF00"

# Texto superior esquerdo:
# Museu do RARU Caso: #...
CASE_OVERLAY_FONT_COLOR_HEX = "#FFFF00"

OVERLAY_BORDER_W_RATIO = 0.09
OVERLAY_BOX_BORDER_W_RATIO = 0.45

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_LOG = LOGS_DIR / f"MuseuCutter_{RUN_ID}.log"

print("=" * 72)
print("RARU REVIVAL / MUSEU DO RARU — CUTTER")
print("=" * 72)
print(f"Raiz:            {RARU_REVIVAL_ROOT}")
print(f"Lives:           {LIVES_DIR}")
print(f"Manifests:       {MANIFESTS_DIR}")
print(f"Cortes:          {CUTS_DIR}")
print(f"Registro casos:  {CASE_REGISTRY_FILE}")
print(f"Log:             {RUN_LOG}")
print()
print("Estrutura pronta.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RARU REVIVAL / MUSEU DO RARU — CUTTER
Raiz:            /content/drive/MyDrive/RaruRevival
Lives:           /content/drive/MyDrive/RaruRevival/Lives
Manifests:       /content/drive/MyDrive/RaruRevival/Manifests
Cortes:          /content/drive/MyDrive/RaruRevival/Cuts
Registro casos:  /content/drive/MyDrive/RaruRevival/Checkpoints/MuseuCaseRegistry/case_registry.txt
Log:             /content/drive/MyDrive/RaruRevival/Logs/MuseuCutter_20261007_201743.log

Estrutura pronta.


In [ ]:
# ============================================================
# CÉLULA 2 — MOTOR DO CUTTER
# ============================================================

import json
import re
import subprocess
import unicodedata
import tempfile
from fractions import Fraction
from PIL import ImageFont

try:
    from zoneinfo import ZoneInfo
except Exception:
    ZoneInfo = None


def log(message=""):
    """Imprime e registra no arquivo de log da execução."""
    text = str(message)
    print(text)
    with open(RUN_LOG, "a", encoding="utf-8") as f:
        f.write(text + "\n")


def ensure_ffmpeg():
    """Garante que ffmpeg e ffprobe estejam disponíveis."""
    if shutil.which("ffmpeg") and shutil.which("ffprobe"):
        log("FFmpeg/ffprobe encontrados.")
        return

    log("FFmpeg não encontrado. Instalando...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=True)
    log("FFmpeg instalado.")


def timestamp_to_seconds(timestamp):
    """Converte HH:MM:SS.mmm em segundos."""
    if not isinstance(timestamp, str):
        raise ValueError(f"Timestamp inválido: {timestamp!r}")

    match = re.fullmatch(r"(\d+):([0-5]\d):([0-5]\d(?:\.\d{1,6})?)", timestamp.strip())
    if not match:
        raise ValueError(f"Timestamp inválido: {timestamp}")

    hours = int(match.group(1))
    minutes = int(match.group(2))
    seconds = float(match.group(3))
    return hours * 3600 + minutes * 60 + seconds


def seconds_to_timestamp(seconds):
    """Converte segundos em HH:MM:SS.mmm."""
    seconds = max(0.0, float(seconds))
    hours = int(seconds // 3600)
    seconds -= hours * 3600
    minutes = int(seconds // 60)
    seconds -= minutes * 60
    return f"{hours:02d}:{minutes:02d}:{seconds:06.3f}"


def sanitize_filename(text, max_length=90):
    """Cria slug seguro para nome de arquivo."""
    text = unicodedata.normalize("NFKD", str(text))
    text = text.encode("ascii", "ignore").decode("ascii")
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", "-", text).strip("-")
    return (text or "momento")[:max_length].rstrip("-")


def sanitize_registry_field(text):
    """Limpa texto para gravação consistente no registro de casos."""
    if text is None:
        return ""
    text = str(text).replace("\r", " ").replace("\n", " ")
    text = text.replace(">>", ">")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def timestamp_for_filename(timestamp):
    """02:18:32.400 -> 02-18-32"""
    seconds = timestamp_to_seconds(timestamp)
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours:02d}-{minutes:02d}-{secs:02d}"


def ffprobe(path):
    cmd = [
        "ffprobe",
        "-v", "error",
        "-show_format",
        "-show_streams",
        "-of", "json",
        str(path),
    ]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        check=True,
    )

    return json.loads(result.stdout)


def get_video_info(path):
    data = ffprobe(path)

    video_stream = next(
        (s for s in data.get("streams", []) if s.get("codec_type") == "video"),
        None,
    )
    audio_stream = next(
        (s for s in data.get("streams", []) if s.get("codec_type") == "audio"),
        None,
    )

    if video_stream is None:
        raise RuntimeError(f"Nenhum stream de vídeo encontrado em: {path}")

    duration = float(data.get("format", {}).get("duration", 0) or 0)
    width = video_stream.get("width")
    height = video_stream.get("height")

    fps_raw = video_stream.get("avg_frame_rate") or video_stream.get("r_frame_rate")
    try:
        fps = float(Fraction(fps_raw)) if fps_raw and fps_raw != "0/0" else None
    except Exception:
        fps = None

    return {
        "duration": duration,
        "width": width,
        "height": height,
        "resolution": f"{width}x{height}",
        "fps": fps,
        "video_codec": video_stream.get("codec_name"),
        "pixel_format": video_stream.get("pix_fmt"),
        "audio_codec": audio_stream.get("codec_name") if audio_stream else None,
        "has_audio": audio_stream is not None,
    }


def load_manifest(path):
    with open(path, "r", encoding="utf-8-sig") as f:
        manifest = json.load(f)

    if not isinstance(manifest, dict):
        raise ValueError("O manifest deve conter um objeto JSON na raiz.")

    moments = manifest.get("moments")

    if not isinstance(moments, list):
        raise ValueError("O manifest não contém uma lista 'moments'.")

    if not moments:
        raise ValueError("O manifest possui 'moments', mas a lista está vazia.")

    return manifest


def ensure_case_registry():
    CASE_REGISTRY_DIR.mkdir(parents=True, exist_ok=True)
    if not CASE_REGISTRY_FILE.exists():
        CASE_REGISTRY_FILE.touch()


def load_case_registry():
    ensure_case_registry()
    records = []

    with open(CASE_REGISTRY_FILE, "r", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line:
                continue

            parts = [part.strip() for part in line.split(">>")]
            if len(parts) < 6:
                continue

            try:
                case_number = int(parts[0])
            except Exception:
                continue

            record = {
                "case_number": case_number,
                "live": parts[1],
                "game": parts[2],
                "id": parts[3],
                "origin": parts[4],
                "created_at": parts[5],
            }
            records.append(record)

    return records


def current_registry_timestamp():
    if ZoneInfo is not None:
        return datetime.now(ZoneInfo("America/Sao_Paulo")).isoformat(timespec="seconds")
    return datetime.now().isoformat(timespec="seconds")


def get_or_create_case_number(live, game, moment_id, origin):
    """
    Reutiliza um caso já registrado se live/game/id/origin coincidirem.
    Caso contrário, cria um novo número sequencial a partir de 0.
    """
    ensure_case_registry()

    live = sanitize_registry_field(live)
    game = sanitize_registry_field(game)
    moment_id = sanitize_registry_field(moment_id)
    origin = sanitize_registry_field(origin or "CURATION") or "CURATION"

    records = load_case_registry()

    for record in records:
        if (
            record["live"] == live
            and record["game"] == game
            and record["id"] == moment_id
            and record["origin"] == origin
        ):
            return record["case_number"], False

    next_case_number = max((r["case_number"] for r in records), default=-1) + 1
    created_at = current_registry_timestamp()

    line = f"{next_case_number} >> {live} >> {game} >> {moment_id} >> {origin} >> {created_at}\n"
    with open(CASE_REGISTRY_FILE, "a", encoding="utf-8") as f:
        f.write(line)

    return next_case_number, True


def validate_manifest(manifest, source_info):
    """Retorna (erros, avisos)."""
    errors = []
    warnings = []
    video_duration = source_info["duration"]

    required_root_fields = ["live", "game", "moments"]
    for field in required_root_fields:
        if field not in manifest or manifest.get(field) in [None, ""]:
            errors.append(f"Campo obrigatório ausente na raiz do manifest: '{field}'.")

    required_moment_fields = ["id", "rank", "title", "start", "end", "origin"]
    ranks_seen = set()

    for index, moment in enumerate(manifest["moments"], start=1):
        label = moment.get("id", f"momento #{index}")

        for field in required_moment_fields:
            if field not in moment:
                errors.append(f"{label}: campo obrigatório ausente: '{field}'.")

        if "rank" in moment:
            try:
                rank = int(moment["rank"])
                if rank in ranks_seen:
                    errors.append(f"{label}: rank duplicado: {rank}.")
                ranks_seen.add(rank)
            except Exception:
                errors.append(f"{label}: rank inválido: {moment['rank']!r}.")

        origin = str(moment.get("origin", "")).strip().upper()
        if origin == "VIEWER" and not moment.get("indicated_by"):
            errors.append(f"{label}: origin=VIEWER exige o campo 'indicated_by'.")

        if "start" not in moment or "end" not in moment:
            continue

        try:
            start = timestamp_to_seconds(moment["start"])
            end = timestamp_to_seconds(moment["end"])
        except Exception as exc:
            errors.append(f"{label}: {exc}")
            continue

        if end <= start:
            errors.append(f"{label}: END precisa ser maior que START.")

        if start >= video_duration:
            errors.append(
                f"{label}: START {moment['start']} está fora do vídeo "
                f"({seconds_to_timestamp(video_duration)})."
            )

        if end > video_duration + 0.250:
            errors.append(
                f"{label}: END {moment['end']} está fora do vídeo "
                f"({seconds_to_timestamp(video_duration)})."
            )

        calculated = end - start

        if "duration_seconds" in moment:
            try:
                declared = float(moment["duration_seconds"])
                difference = abs(calculated - declared)

                if difference > 0.100:
                    warnings.append(
                        f"{label}: duration_seconds={declared:.3f}s, "
                        f"mas START/END resultam em {calculated:.3f}s."
                    )
            except Exception:
                warnings.append(
                    f"{label}: duration_seconds não pôde ser interpretado."
                )

    return errors, warnings


def make_output_filename(moment):
    rank = int(moment["rank"])
    slug = sanitize_filename(moment["title"])
    ts = timestamp_for_filename(moment["start"])
    return f"MUSEU_{rank:02d}_{ts}_{slug}.mp4"


def normalize_hex_color(value):
    """
    Valida uma cor HEX e converte para o formato do FFmpeg.

    Aceita:
      #FFFFFF
      FFFFFF

    Retorna:
      0xFFFFFF
    """
    if not isinstance(value, str):
        raise ValueError(
            f"Cor HEX inválida: {value!r}. "
            "Use o formato #RRGGBB."
        )

    value = value.strip()

    if value.startswith("#"):
        value = value[1:]

    if not re.fullmatch(r"[0-9A-Fa-f]{6}", value):
        raise ValueError(
            f"Cor HEX inválida: {value!r}. "
            "Use exatamente 6 dígitos, por exemplo #FFFFFF."
        )

    return f"0x{value.upper()}"


def escape_drawtext_path(path):
    """Escapa um caminho de arquivo para o filtro drawtext do FFmpeg."""
    text = str(path)
    text = text.replace("\\", r"\\")
    text = text.replace(":", r"\:")
    text = text.replace("'", r"\'")
    text = text.replace(",", r"\,")
    text = text.replace(";", r"\;")
    text = text.replace("[", r"\[")
    text = text.replace("]", r"\]")
    return text


def load_overlay_font(size):
    """Carrega a mesma fonte usada pelo FFmpeg para medir o texto."""
    if OVERLAY_FONT_PATH.exists():
        return ImageFont.truetype(str(OVERLAY_FONT_PATH), size=size)

    # Fallback normalmente desnecessário no Colab.
    return ImageFont.load_default()


def measure_text_block(lines, font_size):
    """
    Mede o bloco de texto considerando:
    - uma linha física por campo;
    - espaçamento entre linhas;
    - padding da caixa.
    """
    font = load_overlay_font(font_size)

    widths = []
    for line in lines:
        bbox = font.getbbox(str(line))
        widths.append(max(0, bbox[2] - bbox[0]))

    # Usa uma altura consistente de linha.
    sample_bbox = font.getbbox("Ag")
    line_height = max(1, sample_bbox[3] - sample_bbox[1])

    line_spacing = max(
        1,
        round(font_size * OVERLAY_LINE_SPACING_RATIO)
    )

    box_border = max(
        1,
        round(font_size * OVERLAY_BOX_BORDER_W_RATIO)
    )

    text_width = max(widths, default=0)
    text_height = (
        len(lines) * line_height
        + max(0, len(lines) - 1) * line_spacing
    )

    # Inclui o padding visual do box do drawtext.
    total_width = text_width + 2 * box_border
    total_height = text_height + 2 * box_border

    return {
        "text_width": text_width,
        "text_height": text_height,
        "total_width": total_width,
        "total_height": total_height,
        "line_height": line_height,
        "line_spacing": line_spacing,
        "box_border": box_border,
    }


def choose_overlay_font_size(lines, video_width, video_height):
    """
    Escolhe o maior tamanho de fonte que mantém TODO o bloco em:
      largura <= 25% do vídeo
      altura  <= 25% do vídeo

    Cada item de `lines` continua sendo uma linha própria.
    """
    max_width = max(1, int(video_width * OVERLAY_MAX_WIDTH_RATIO))
    max_height = max(1, int(video_height * OVERLAY_MAX_HEIGHT_RATIO))

    desired_size = max(
        OVERLAY_MIN_FONT_SIZE,
        round(
            OVERLAY_BASE_FONT_SIZE_1080
            * (video_height / 1080.0)
        )
    )

    for font_size in range(desired_size, OVERLAY_MIN_FONT_SIZE - 1, -1):
        metrics = measure_text_block(lines, font_size)

        if (
            metrics["total_width"] <= max_width
            and metrics["total_height"] <= max_height
        ):
            return font_size, metrics, max_width, max_height

    # Segurança extrema: continua reduzindo até 1 px.
    for font_size in range(OVERLAY_MIN_FONT_SIZE - 1, 0, -1):
        metrics = measure_text_block(lines, font_size)

        if (
            metrics["total_width"] <= max_width
            and metrics["total_height"] <= max_height
        ):
            return font_size, metrics, max_width, max_height

    raise RuntimeError(
        "Não foi possível dimensionar o overlay dentro do limite de 25%."
    )


def create_overlay_text_file(lines):
    """
    Cria um arquivo UTF-8 com QUEBRAS DE LINHA REAIS.

    Isso evita o comportamento ambíguo de '\\n' dentro de drawtext
    e garante que Live, Game e 'Momento escolhido por' sejam
    renderizados em linhas físicas separadas.
    """
    temp = tempfile.NamedTemporaryFile(
        mode="w",
        suffix=".txt",
        prefix="museu_overlay_",
        encoding="utf-8",
        delete=False,
    )

    try:
        temp.write("\n".join(str(line) for line in lines))
        temp.flush()
        return Path(temp.name)
    finally:
        temp.close()


def build_drawtext_filter(
    text_file,
    font_size,
    metrics,
    position,
    font_color_hex,
):
    """Monta um drawtext com os limites já calculados."""
    font_color = normalize_hex_color(font_color_hex)

    font_prefix = ""
    if OVERLAY_FONT_PATH.exists():
        font_prefix = (
            f"fontfile='{escape_drawtext_path(OVERLAY_FONT_PATH)}':"
        )

    border_w = max(
        1,
        round(font_size * OVERLAY_BORDER_W_RATIO)
    )

    margin = "max(8\\,round(h*%s))" % OVERLAY_MARGIN_RATIO

    common = (
        f"{font_prefix}"
        f"textfile='{escape_drawtext_path(text_file)}':"
        f"reload=0:"
        f"fontsize={font_size}:"
        f"fontcolor={font_color}:"
        f"borderw={border_w}:"
        f"bordercolor=black:"
        f"line_spacing={metrics['line_spacing']}:"
        f"box=1:"
        f"boxcolor=black@{OVERLAY_BOX_OPACITY}:"
        f"boxborderw={metrics['box_border']}"
    )

    if position == "bottom_left":
        return (
            "drawtext="
            f"{common}:"
            f"x={margin}:"
            f"y=h-th-{margin}"
        )

    if position == "top_left":
        return (
            "drawtext="
            f"{common}:"
            f"x={margin}:"
            f"y={margin}"
        )

    raise ValueError(f"Posição de overlay desconhecida: {position}")


def prepare_overlay(manifest, moment, case_number, source_info):
    """
    Prepara os dois overlays e retorna:
      filtro FFmpeg,
      arquivos temporários,
      metadados de dimensionamento.
    """
    live = str(manifest.get("live", "")).strip()
    game = str(manifest.get("game", "")).strip()
    origin = str(moment.get("origin", "CURATION")).strip().upper()
    indicated_by = str(moment.get("indicated_by") or "").strip()

    # IMPORTANTE:
    # Cada campo é um item separado, portanto uma linha física própria.
    bottom_lines = [
        f"Live: {live}",
        f"Game: {game}",
    ]

    if origin == "VIEWER":
        bottom_lines.append(
            f"Momento escolhido por {indicated_by}"
        )

    top_lines = [
        f"Museu do RARU Caso: #{case_number}"
    ]

    video_width = int(source_info["width"])
    video_height = int(source_info["height"])

    bottom_font_size, bottom_metrics, bottom_max_w, bottom_max_h = (
        choose_overlay_font_size(
            bottom_lines,
            video_width,
            video_height,
        )
    )

    top_font_size, top_metrics, top_max_w, top_max_h = (
        choose_overlay_font_size(
            top_lines,
            video_width,
            video_height,
        )
    )

    bottom_file = create_overlay_text_file(bottom_lines)
    top_file = create_overlay_text_file(top_lines)

    filters = [
        build_drawtext_filter(
            bottom_file,
            bottom_font_size,
            bottom_metrics,
            "bottom_left",
            BOTTOM_OVERLAY_FONT_COLOR_HEX,
        ),
        build_drawtext_filter(
            top_file,
            top_font_size,
            top_metrics,
            "top_left",
            CASE_OVERLAY_FONT_COLOR_HEX,
        ),
    ]

    metadata = {
        "bottom_left": {
            "lines": bottom_lines,
            "font_size": bottom_font_size,
            "font_color_hex": BOTTOM_OVERLAY_FONT_COLOR_HEX,
            "measured_width": bottom_metrics["total_width"],
            "measured_height": bottom_metrics["total_height"],
            "max_width": bottom_max_w,
            "max_height": bottom_max_h,
        },
        "top_left": {
            "lines": top_lines,
            "font_size": top_font_size,
            "font_color_hex": CASE_OVERLAY_FONT_COLOR_HEX,
            "measured_width": top_metrics["total_width"],
            "measured_height": top_metrics["total_height"],
            "max_width": top_max_w,
            "max_height": top_max_h,
        },
    }

    return ",".join(filters), [bottom_file, top_file], metadata


def cut_moment(video_path, manifest, moment, output_dir, source_info, case_number):
    start = timestamp_to_seconds(moment["start"])
    end = timestamp_to_seconds(moment["end"])
    duration = end - start

    filename = make_output_filename(moment)
    output_path = output_dir / filename

    result_base = {
        "id": moment.get("id"),
        "rank": int(moment["rank"]),
        "title": moment["title"],
        "live": manifest.get("live"),
        "game": manifest.get("game"),
        "origin": moment.get("origin"),
        "case_number": case_number,
        "file": filename,
        "start": moment["start"],
        "end": moment["end"],
    }

    if output_path.exists() and not OVERWRITE:
        log(f"  ↳ Pulado: {filename} já existe.")
        result_base["status"] = "PULADO_EXISTENTE"
        return result_base

    overlay_filter, overlay_temp_files, overlay_metadata = prepare_overlay(
        manifest=manifest,
        moment=moment,
        case_number=case_number,
        source_info=source_info,
    )

    log(
        "  Overlay inferior: "
        f"{len(overlay_metadata['bottom_left']['lines'])} linha(s), "
        f"fonte {overlay_metadata['bottom_left']['font_size']} px, "
        f"{overlay_metadata['bottom_left']['measured_width']}x"
        f"{overlay_metadata['bottom_left']['measured_height']} px "
        f"(limite {overlay_metadata['bottom_left']['max_width']}x"
        f"{overlay_metadata['bottom_left']['max_height']} px)"
    )

    overwrite_flag = "-y" if OVERWRITE else "-n"

    cmd = [
        "ffmpeg",
        "-hide_banner",
        "-loglevel", "error",
        overwrite_flag,
        "-ss", f"{start:.6f}",
        "-i", str(video_path),
        "-t", f"{duration:.6f}",
        "-map", "0:v:0",
        "-map", "0:a:0?",
        "-map_metadata", "0",
        "-vf", overlay_filter,
        "-c:v", "libx264",
        "-preset", PRESET,
        "-crf", str(VIDEO_CRF),
        "-fps_mode", "passthrough",
        "-c:a", "aac",
        "-b:a", AUDIO_BITRATE,
        "-movflags", "+faststart",
        "-avoid_negative_ts", "make_zero",
        str(output_path),
    ]

    try:
        subprocess.run(cmd, check=True)
    finally:
        for temp_file in overlay_temp_files:
            try:
                temp_file.unlink(missing_ok=True)
            except Exception:
                pass

    output_info = get_video_info(output_path)
    actual_duration = output_info["duration"]
    duration_error = abs(actual_duration - duration)

    same_resolution = (
        output_info["width"] == source_info["width"]
        and output_info["height"] == source_info["height"]
    )

    fps_ok = True
    if source_info["fps"] and output_info["fps"]:
        fps_ok = abs(source_info["fps"] - output_info["fps"]) < 0.10

    duration_ok = duration_error <= 0.200
    status = "OK" if same_resolution and fps_ok and duration_ok else "VERIFICAR"

    result_base.update({
        "score": moment.get("score"),
        "category": moment.get("category", []),
        "target_duration_seconds": round(duration, 3),
        "actual_duration_seconds": round(actual_duration, 3),
        "duration_error_ms": round(duration_error * 1000, 2),
        "resolution": output_info["resolution"],
        "fps": round(output_info["fps"], 3) if output_info["fps"] else None,
        "video_codec": output_info["video_codec"],
        "overlay": overlay_metadata,
        "status": status,
    })

    return result_base


def process_job(job):
    video_path = job["video_path"]
    manifest_path = job["manifest_path"]
    output_dir = job["output_dir"]
    output_dir.mkdir(parents=True, exist_ok=True)

    log("")
    log("=" * 72)
    log(f"PROCESSANDO: {job['source_name']}")
    log("=" * 72)
    log(f"Vídeo:    {video_path}")
    log(f"Manifest: {manifest_path}")
    log(f"Saída:    {output_dir}")

    manifest = load_manifest(manifest_path)
    source_info = get_video_info(video_path)

    if source_info["fps"]:
        log(
            f"Fonte: {source_info['resolution']} | "
            f"{source_info['fps']:.3f} FPS | "
            f"{seconds_to_timestamp(source_info['duration'])}"
        )
    else:
        log(
            f"Fonte: {source_info['resolution']} | FPS desconhecido | "
            f"{seconds_to_timestamp(source_info['duration'])}"
        )

    errors, warnings = validate_manifest(manifest, source_info)

    for warning in warnings:
        log(f"AVISO: {warning}")

    if errors:
        for error in errors:
            log(f"ERRO: {error}")
        raise RuntimeError(
            f"Manifest inválido para '{job['source_name']}'. "
            f"{len(errors)} erro(s) encontrado(s)."
        )

    moments = sorted(manifest["moments"], key=lambda x: int(x["rank"]))
    cuts_report = []

    for moment in moments:
        rank = int(moment["rank"])
        origin = str(moment.get("origin", "CURATION")).strip().upper() or "CURATION"
        case_number, created_now = get_or_create_case_number(
            live=manifest.get("live"),
            game=manifest.get("game"),
            moment_id=moment.get("id"),
            origin=origin,
        )

        log("")
        log(
            f"[{rank:02d}] Caso #{case_number} | {moment['title']} | "
            f"{moment['start']} → {moment['end']}"
        )
        log(f"  Origem: {origin} | Registro: {'novo' if created_now else 'existente'}")

        try:
            result = cut_moment(
                video_path=video_path,
                manifest=manifest,
                moment=moment,
                output_dir=output_dir,
                source_info=source_info,
                case_number=case_number,
            )
            cuts_report.append(result)

            if result["status"] == "PULADO_EXISTENTE":
                continue

            log(
                f"  ↳ {result['file']} | "
                f"{result['actual_duration_seconds']:.3f}s | "
                f"{result['resolution']} | "
                f"{result['fps']} FPS | "
                f"{result['status']}"
            )

        except Exception as exc:
            error_result = {
                "id": moment.get("id"),
                "rank": rank,
                "title": moment.get("title"),
                "live": manifest.get("live"),
                "game": manifest.get("game"),
                "origin": origin,
                "case_number": case_number,
                "start": moment.get("start"),
                "end": moment.get("end"),
                "status": "ERRO",
                "error": str(exc),
            }
            cuts_report.append(error_result)
            log(f"  ↳ ERRO: {exc}")

            if not CONTINUE_ON_ERROR:
                raise

    final_report = {
        "project": manifest.get("project", "Museu do Raru"),
        "live": manifest.get("live"),
        "game": manifest.get("game"),
        "video": str(video_path),
        "manifest": str(manifest_path),
        "case_registry_file": str(CASE_REGISTRY_FILE),
        "source_info": source_info,
        "settings": {
            "video_encoder": "libx264",
            "crf": VIDEO_CRF,
            "preset": PRESET,
            "audio_codec": "aac",
            "audio_bitrate": AUDIO_BITRATE,
            "overlay_font": str(OVERLAY_FONT_PATH),
        },
        "cuts": cuts_report,
    }

    report_path = output_dir / "MUSEU_CUT_REPORT.json"
    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(final_report, f, indent=2, ensure_ascii=False)

    zip_path = None
    if CREATE_ZIP:
        zip_base = output_dir.parent / f"{output_dir.name}_MUSEU"
        zip_path = Path(
            shutil.make_archive(
                str(zip_base),
                "zip",
                root_dir=output_dir,
            )
        )

    ok_count = sum(1 for x in cuts_report if x.get("status") == "OK")
    verify_count = sum(1 for x in cuts_report if x.get("status") == "VERIFICAR")
    skipped_count = sum(1 for x in cuts_report if x.get("status") == "PULADO_EXISTENTE")
    error_count = sum(1 for x in cuts_report if x.get("status") == "ERRO")

    log("")
    log(
        f"Resultado: {ok_count} OK | {verify_count} verificar | "
        f"{skipped_count} pulados | {error_count} erros"
    )
    log(f"Relatório: {report_path}")
    if zip_path:
        log(f"ZIP:       {zip_path}")

    return {
        "source_name": job["source_name"],
        "report_path": str(report_path),
        "zip_path": str(zip_path) if zip_path else None,
        "cuts": cuts_report,
    }


ensure_ffmpeg()
log("Motor do Museu Cutter carregado.")


FFmpeg/ffprobe encontrados.
Motor do Museu Cutter carregado.


In [ ]:
# ============================================================
# CÉLULA 3 — DESCOBRIR LIVES E ASSOCIAR MANIFESTS
# ============================================================
#
# REGRA IMPORTANTE:
# 1. Primeiro procura vídeos.
# 2. Se não encontrar nenhum vídeo, NÃO procura manifests.
# 3. Somente depois associa cada vídeo ao JSON de mesmo nome-base.
# ============================================================

video_files = sorted(
    [
        file
        for file in LIVES_DIR.iterdir()
        if file.is_file() and file.suffix.lower() in VIDEO_EXTENSIONS
    ],
    key=lambda p: p.name.casefold(),
)

HAS_VIDEOS = bool(video_files)
jobs = []
videos_without_manifest = []

log("")
log("=" * 72)
log("DESCOBERTA DE LIVES")
log("=" * 72)

if not HAS_VIDEOS:
    log(f"Nenhum vídeo encontrado em: {LIVES_DIR}")
    log("A busca por manifests NÃO será executada.")
    log("Execução encerrada antes da etapa de manifests.")

else:
    log(f"{len(video_files)} live(s) encontrada(s):")
    for i, video in enumerate(video_files, start=1):
        log(f"  {i:02d}. {video.name}")

    log("")
    log("=" * 72)
    log("ASSOCIAÇÃO LIVE ↔ MANIFEST")
    log("=" * 72)

    for video_path in video_files:
        source_name = video_path.stem

        # Correspondência EXATA pelo nome-base:
        # Lives/Amanda 1.mp4 -> Manifests/Amanda 1.json
        manifest_path = MANIFESTS_DIR / f"{source_name}.json"

        if not manifest_path.is_file():
            videos_without_manifest.append(video_path)
            log(f"SEM MANIFEST: {video_path.name} → esperado: {manifest_path.name}")
            continue

        output_dir = CUTS_DIR / source_name

        jobs.append(
            {
                "source_name": source_name,
                "video_path": video_path,
                "manifest_path": manifest_path,
                "output_dir": output_dir,
            }
        )

        log(f"PRONTO: {video_path.name} ↔ {manifest_path.name}")

    log("")
    log(f"Jobs prontos: {len(jobs)}")
    log(f"Lives sem manifest: {len(videos_without_manifest)}")

    if not jobs:
        log("Nenhum par vídeo/manifest encontrado. Nenhum corte será realizado.")



DESCOBERTA DE LIVES
1 live(s) encontrada(s):
  01. Amanda 1.mp4

ASSOCIAÇÃO LIVE ↔ MANIFEST
PRONTO: Amanda 1.mp4 ↔ Amanda 1.json

Jobs prontos: 1
Lives sem manifest: 0


In [ ]:
# ============================================================
# CÉLULA 4 — PROCESSAR TODOS OS JOBS PRONTOS
# ============================================================

all_results = []

if not HAS_VIDEOS:
    print("Nada a processar: nenhuma live foi encontrada.")

elif not jobs:
    print("Nada a processar: nenhuma live possui manifest correspondente.")

else:
    print("=" * 72)
    print(f"INICIANDO {len(jobs)} JOB(S)")
    print("=" * 72)

    for job in jobs:
        try:
            result = process_job(job)
            all_results.append(result)
        except Exception as exc:
            log("")
            log(f"FALHA NO JOB '{job['source_name']}': {exc}")

            all_results.append(
                {
                    "source_name": job["source_name"],
                    "status": "ERRO_JOB",
                    "error": str(exc),
                }
            )

            if not CONTINUE_ON_ERROR:
                raise

    print()
    print("=" * 72)
    print("RESUMO FINAL")
    print("=" * 72)

    for result in all_results:
        if result.get("status") == "ERRO_JOB":
            print(f"❌ {result['source_name']}: erro no job")
            continue

        cuts = result.get("cuts", [])
        ok = sum(1 for x in cuts if x.get("status") == "OK")
        verify = sum(1 for x in cuts if x.get("status") == "VERIFICAR")
        skipped = sum(1 for x in cuts if x.get("status") == "PULADO_EXISTENTE")
        errors = sum(1 for x in cuts if x.get("status") == "ERRO")

        print(
            f"✅ {result['source_name']}: "
            f"{ok} OK | {verify} verificar | {skipped} pulados | {errors} erros"
        )

    print()
    print(f"Log completo: {RUN_LOG}")
    print(f"Registro de casos: {CASE_REGISTRY_FILE}")


INICIANDO 1 JOB(S)

PROCESSANDO: Amanda 1
Vídeo:    /content/drive/MyDrive/RaruRevival/Lives/Amanda 1.mp4
Manifest: /content/drive/MyDrive/RaruRevival/Manifests/Amanda 1.json
Saída:    /content/drive/MyDrive/RaruRevival/Cuts/Amanda 1
Fonte: 1280x720 | 30.000 FPS | 03:16:15.884

[01] Caso #0 | A resposta errada vira uma morte bizarra | 02:18:32.400 → 02:19:14.800
  Origem: CURATION | Registro: existente
  Overlay inferior: 2 linha(s), fonte 24 px, 383x48 px (limite 640x360 px)
  ↳ MUSEU_01_02-18-32_a-resposta-errada-vira-uma-morte-bizarra.mp4 | 42.423s | 1280x720 | 30.0 FPS | OK

[02] Caso #1 | O porco surge do nada | 02:12:54.920 → 02:13:15.820
  Origem: CURATION | Registro: existente
  Overlay inferior: 2 linha(s), fonte 24 px, 383x48 px (limite 640x360 px)
  ↳ MUSEU_02_02-12-54_o-porco-surge-do-nada.mp4 | 20.923s | 1280x720 | 30.0 FPS | OK

[03] Caso #2 | Eu ouvi um barulho... | 01:41:14.500 → 01:41:43.020
  Origem: CURATION | Registro: existente
  Overlay inferior: 2 linha(s), fonte

## Onde colocar os manifests

Use:

```text
/content/drive/MyDrive/RaruRevival/Manifests/
```

O nome-base deve ser igual ao vídeo.

Exemplos válidos:

```text
Lives/Amanda 1.mp4
Manifests/Amanda 1.json
```

```text
Lives/Silent Hill - Parte 03.mkv
Manifests/Silent Hill - Parte 03.json
```

A extensão do vídeo pode variar; o nome-base é o que determina a associação.

## Registro de casos

O registro persistente fica em:

```text
RaruRevival/Checkpoints/MuseuCaseRegistry/case_registry.txt
```

Cada linha segue este formato:

```text
numero_caso >> live >> game >> id >> origin >> data_do_registro
```

O primeiro caso começa em `0`. Se um momento com o mesmo `live + game + id + origin` já estiver registrado, o número é reutilizado.

## Saída

Para `Amanda 1.mp4`:

```text
RaruRevival/Cuts/Amanda 1/
├── MUSEU_01_02-18-32_a-resposta-errada-vira-uma-morte-bizarra.mp4
├── MUSEU_02_02-12-54_o-porco-surge-do-nada.mp4
├── ...
└── MUSEU_CUT_REPORT.json
```

Também é criado, se `CREATE_ZIP = True`:

```text
RaruRevival/Cuts/Amanda 1_MUSEU.zip
```

Os logs da execução ficam em:

```text
RaruRevival/Logs/MuseuCutter_YYYYMMDD_HHMMSS.log
```


## Limite dos overlays

Os dois blocos de overlay são medidos usando a fonte real antes do FFmpeg executar.

- `Live` fica em uma linha.
- `Game` fica em outra linha.
- `Momento escolhido por ...` fica em uma terceira linha quando a origem é `VIEWER`.
- Nenhum bloco pode exceder 50% da largura ou 50% da altura do vídeo.
- Se necessário, o tamanho da fonte é reduzido automaticamente.
- As quebras de linha são gravadas em arquivos temporários UTF-8 e lidas pelo `drawtext`, evitando que `\n` apareça literalmente no vídeo.


## Configuração de cor e posição

Na primeira célula:

```python
BOTTOM_OVERLAY_FONT_COLOR_HEX = "#FFFFFF"
CASE_OVERLAY_FONT_COLOR_HEX = "#FFFFFF"
```

- `BOTTOM_OVERLAY_FONT_COLOR_HEX`: cor de `Live`, `Game` e `Momento escolhido por`.
- `CASE_OVERLAY_FONT_COLOR_HEX`: cor de `Museu do RARU Caso: #...`.
- As cores aceitam `#RRGGBB` ou `RRGGBB`.
- A mensagem do caso fica no **canto superior esquerdo**.
